In [11]:
# Cell 1: Force TensorFlow to use CPU only (MUST run BEFORE importing tensorflow)
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("✅ GPU disabled (CUDA_VISIBLE_DEVICES = -1). TensorFlow will run on CPU.")
print("📍 Working directory:", os.getcwd())


✅ GPU disabled (CUDA_VISIBLE_DEVICES = -1). TensorFlow will run on CPU.
📍 Working directory: /workspace


In [12]:
# Cell 2: Import libraries + confirm devices
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("✅ Imports OK")
print("🧠 TensorFlow version:", tf.__version__)

cpus = tf.config.list_physical_devices("CPU")
gpus = tf.config.list_physical_devices("GPU")
print(f"🖥️ CPUs visible to TF: {len(cpus)}")
print(f"🎮 GPUs visible to TF: {len(gpus)}")
print("✅ Running on CPU only." if len(gpus) == 0 else "⚠️ GPU is visible — restart kernel and run Cell 1 first.")


✅ Imports OK
🧠 TensorFlow version: 2.15.1
🖥️ CPUs visible to TF: 1
🎮 GPUs visible to TF: 0
✅ Running on CPU only.


In [13]:
# Cell 3: Configure LOCAL paths (edit if your folder names differ)
import os
from pathlib import Path

LOCAL_ROOT = Path("tight_loose_clothing_augmented/data/lower body clothing categories").resolve()
MODEL_DIR  = Path("models").resolve()
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Paths configured")
print("📂 Dataset root:", LOCAL_ROOT)
print("💾 Model dir:    ", MODEL_DIR)

if not LOCAL_ROOT.exists():
    raise FileNotFoundError(f"❌ Dataset folder not found: {LOCAL_ROOT}")


✅ Paths configured
📂 Dataset root: /workspace/tight_loose_clothing_augmented/data/lower body clothing categories
💾 Model dir:     /workspace/models


In [14]:
# Cell 4: Verify dataset + count images per class
print("🔍 Verifying dataset...\n")

categories = sorted([p.name for p in LOCAL_ROOT.iterdir() if p.is_dir()])
print(f"✅ Found {len(categories)} categories")

valid_ext = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp")
total_images = 0

for i, cat in enumerate(categories, 1):
    cat_path = LOCAL_ROOT / cat
    imgs = [f for f in cat_path.iterdir() if f.is_file() and f.suffix.lower() in valid_ext]
    count = len(imgs)
    total_images += count
    print(f"   {i:02d}. {cat}: {count} images")

print("\n📊 Total images:", total_images)

if len(categories) == 0:
    raise ValueError("❌ No class folders found inside LOCAL_ROOT.")
if total_images == 0:
    raise ValueError("❌ No images found. Check extensions and folder structure.")


🔍 Verifying dataset...

✅ Found 5 categories
   01. cargo pants: 200 images
   02. culottes: 200 images
   03. shorts: 200 images
   04. skirt: 200 images
   05. trousers: 200 images

📊 Total images: 1000


In [15]:
# Cell 5: Training parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 300
NUM_CLASSES = len(categories)

print("⚙️ Training config")
print("   IMG_SIZE:", IMG_SIZE)
print("   BATCH_SIZE:", BATCH_SIZE)
print("   EPOCHS:", EPOCHS)
print("   NUM_CLASSES:", NUM_CLASSES)


⚙️ Training config
   IMG_SIZE: (224, 224)
   BATCH_SIZE: 32
   EPOCHS: 300
   NUM_CLASSES: 5


In [16]:
# Cell 6: Create data generators (train/val split)
print("🎨 Creating data generators...")

datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2
)

train_generator = datagen.flow_from_directory(
    str(LOCAL_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    str(LOCAL_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("\n✅ Generators ready")
print("📊 Train samples:", train_generator.samples)
print("📊 Val samples:  ", val_generator.samples)

print("\n🏷️ Class index mapping (IMPORTANT):")
for name, idx in train_generator.class_indices.items():
    print(f"   {idx}: {name}")

gen_num_classes = len(train_generator.class_indices)
print(f"\n🔎 Generator classes: {gen_num_classes} | NUM_CLASSES: {NUM_CLASSES}")
if gen_num_classes != NUM_CLASSES:
    raise ValueError(f"❌ Class count mismatch: generator={gen_num_classes}, NUM_CLASSES={NUM_CLASSES}")


🎨 Creating data generators...
Found 800 images belonging to 5 classes.
Found 200 images belonging to 5 classes.

✅ Generators ready
📊 Train samples: 800
📊 Val samples:   200

🏷️ Class index mapping (IMPORTANT):
   0: cargo pants
   1: culottes
   2: shorts
   3: skirt
   4: trousers

🔎 Generator classes: 5 | NUM_CLASSES: 5


In [17]:
# Cell 7: Quick sanity-check one batch
x_batch, y_batch = next(train_generator)
print("✅ Batch loaded")
print("x_batch:", x_batch.shape, "dtype:", x_batch.dtype, "min/max:", float(x_batch.min()), float(x_batch.max()))
print("y_batch:", y_batch.shape, "dtype:", y_batch.dtype)
print("y_batch sample (first 5 rows):\n", y_batch[:5])


✅ Batch loaded
x_batch: (32, 224, 224, 3) dtype: float32 min/max: 0.0 1.0
y_batch: (32, 5) dtype: float32
y_batch sample (first 5 rows):
 [[0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0.]]


In [18]:
# Cell 8: Build MobileNetV2 transfer-learning model (frozen base)
print("🔧 Loading MobileNetV2 base...")

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

print("✅ Model built")
print("   Base frozen:", all([not l.trainable for l in base_model.layers]))
print("📋 Model summary:")
model.summary()


🔧 Loading MobileNetV2 base...
✅ Model built
   Base frozen: True
📋 Model summary:
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_2[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_rel

In [19]:
# Cell 9: Compile
print("⚙️ Compiling model...")

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
)

print("✅ Compiled")
print("   Optimizer: Adam(lr=1e-4)")
print("   Loss: categorical_crossentropy")
print("   Metric: CategoricalAccuracy")


⚙️ Compiling model...
✅ Compiled
   Optimizer: Adam(lr=1e-4)
   Loss: categorical_crossentropy
   Metric: CategoricalAccuracy


In [20]:
# Cell 10: Callbacks
print("🎯 Setting callbacks...")

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = ReduceLROnPlateau(
    monitor="loss",
    factor=0.1,
    patience=10,
    min_lr=1e-7,
    verbose=1
)

print("✅ Callbacks ready")
print("   - EarlyStopping(patience=20, monitor=val_loss)")
print("   - ReduceLROnPlateau(patience=10, factor=0.1, monitor=val_loss)")


🎯 Setting callbacks...
✅ Callbacks ready
   - EarlyStopping(patience=20, monitor=val_loss)
   - ReduceLROnPlateau(patience=10, factor=0.1, monitor=val_loss)


In [21]:
# Cell 11: Train (frozen base)
print("\n" + "="*60)
print("🚀 TRAINING (PHASE 1: frozen base)")
print("="*60 + "\n")

history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

print("\n✅ Phase 1 training complete")



🚀 TRAINING (PHASE 1: frozen base)

Epoch 1/300
25/25 [==============================] - 20s 686ms/step - loss: 1.8076 - accuracy: 0.2500 - val_loss: 1.3363 - val_accuracy: 0.4550 - lr: 1.0000e-04
Epoch 2/300
25/25 [==============================] - 16s 618ms/step - loss: 1.3314 - accuracy: 0.4550 - val_loss: 1.1973 - val_accuracy: 0.5550 - lr: 1.0000e-04
Epoch 3/300
25/25 [==============================] - 15s 599ms/step - loss: 1.0456 - accuracy: 0.5975 - val_loss: 1.1128 - val_accuracy: 0.5950 - lr: 1.0000e-04
Epoch 4/300
25/25 [==============================] - 15s 587ms/step - loss: 0.8805 - accuracy: 0.6812 - val_loss: 1.0605 - val_accuracy: 0.6100 - lr: 1.0000e-04
Epoch 5/300
25/25 [==============================] - 15s 580ms/step - loss: 0.7724 - accuracy: 0.7362 - val_loss: 1.0370 - val_accuracy: 0.6300 - lr: 1.0000e-04
Epoch 6/300
25/25 [==============================] - 14s 565ms/step - loss: 0.7189 - accuracy: 0.7513 - val_loss: 1.0095 - val_accuracy: 0.6400 - lr: 1.0000e-0

In [22]:
# Cell 12 (Optional): Fine-tune last N layers of MobileNetV2 (often improves accuracy)
# You can skip this cell if you don't want fine-tuning.

print("🔧 Fine-tuning setup...")

FINE_TUNE = True
UNFREEZE_LAST_N = 30  # try 20–60 depending on dataset size
FINE_TUNE_LR = 1e-5
FINE_TUNE_EPOCHS = 50

if not FINE_TUNE:
    print("⏭️ Fine-tuning skipped (FINE_TUNE=False)")
else:
    # Unfreeze last N layers (keep BatchNorm layers frozen for stability)
    for layer in base_model.layers[:-UNFREEZE_LAST_N]:
        layer.trainable = False
    for layer in base_model.layers[-UNFREEZE_LAST_N:]:
        if "batch_normalization" in layer.name.lower():
            layer.trainable = False
        else:
            layer.trainable = True

    print(f"✅ Unfroze last {UNFREEZE_LAST_N} layers (BatchNorm kept frozen)")
    
    model.compile(
        optimizer=Adam(learning_rate=FINE_TUNE_LR),
        loss="categorical_crossentropy",
        metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")]
    )
    print("✅ Re-compiled for fine-tuning")
    print("   Fine-tune LR:", FINE_TUNE_LR)

    print("\n" + "="*60)
    print("🚀 TRAINING (PHASE 2: fine-tuning)")
    print("="*60 + "\n")

    history2 = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=FINE_TUNE_EPOCHS,
        callbacks=[early_stopping, lr_scheduler],
        verbose=1
    )

    print("\n✅ Phase 2 fine-tuning complete")


🔧 Fine-tuning setup...
✅ Unfroze last 30 layers (BatchNorm kept frozen)
✅ Re-compiled for fine-tuning
   Fine-tune LR: 1e-05

🚀 TRAINING (PHASE 2: fine-tuning)

Epoch 1/50
25/25 [==============================] - 20s 640ms/step - loss: 0.9168 - accuracy: 0.6700 - val_loss: 1.0055 - val_accuracy: 0.7050 - lr: 1.0000e-05
Epoch 2/50
25/25 [==============================] - 20s 788ms/step - loss: 0.7322 - accuracy: 0.7262 - val_loss: 1.0565 - val_accuracy: 0.7050 - lr: 1.0000e-05
Epoch 3/50
12/25 [=============>................] - ETA: 6s - loss: 0.6394 - accuracy: 0.7708

KeyboardInterrupt: 

In [23]:
# Cell 13: Save Keras model (H5 + SavedModel)
import time

timestamp = time.strftime("%Y%m%d_%H%M%S")
H5_PATH = MODEL_DIR / f"lower_body_mobilenetv2_{timestamp}.h5"
SAVEDMODEL_DIR = MODEL_DIR / f"lower_body_mobilenetv2_savedmodel_{timestamp}"

print("💾 Saving model...")
print("   H5 path:        ", H5_PATH)
print("   SavedModel dir: ", SAVEDMODEL_DIR)

model.save(H5_PATH)  # .h5 (Keras format)
model.export(str(SAVEDMODEL_DIR))  # SavedModel (TF 2.13+; if your TF is older, see note below)

print("✅ Model saved")
print("   H5 exists:", H5_PATH.exists())
print("   SavedModel exists:", SAVEDMODEL_DIR.exists())


💾 Saving model...
   H5 path:         /workspace/models/lower_body_mobilenetv2_20260202_130936.h5
   SavedModel dir:  /workspace/models/lower_body_mobilenetv2_savedmodel_20260202_130936


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


INFO:tensorflow:Assets written to: /workspace/models/lower_body_mobilenetv2_savedmodel_20260202_130936/assets


INFO:tensorflow:Assets written to: /workspace/models/lower_body_mobilenetv2_savedmodel_20260202_130936/assets


Saved artifact at '/workspace/models/lower_body_mobilenetv2_savedmodel_20260202_130936'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_2')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  128402520497360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396809302288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652390224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652391760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652389648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652390608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652390992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652391376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652390032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128396652389840: TensorSpec(shape=(), 

In [24]:
# Cell 14: Save class order EXACTLY as training generator used (CRITICAL for correct JS inference)
import json

# This is the class order the model trained with:
idx_to_class = {int(v): k for k, v in train_generator.class_indices.items()}
class_names_in_training_order = [idx_to_class[i] for i in range(len(idx_to_class))]

CLASSES_JSON = MODEL_DIR / f"lower_body_classes_{timestamp}.json"
with open(CLASSES_JSON, "w", encoding="utf-8") as f:
    json.dump(class_names_in_training_order, f, indent=2)

print("✅ Saved classes JSON")
print("   Path:", CLASSES_JSON)
print("   Num classes:", len(class_names_in_training_order))
print("   First 10 classes:", class_names_in_training_order[:10])


✅ Saved classes JSON
   Path: /workspace/models/lower_body_classes_20260202_130936.json
   Num classes: 5
   First 10 classes: ['cargo pants', 'culottes', 'shorts', 'skirt', 'trousers']


In [25]:
# Cell 15: Install tensorflowjs converter (if not installed)
print("📦 Installing / verifying tensorflowjs...")

import sys
!{sys.executable} -m pip install -q tensorflowjs

print("✅ tensorflowjs installed / available")


📦 Installing / verifying tensorflowjs...
✅ tensorflowjs installed / available


In [26]:
# Cell 16: Convert Keras .h5 -> TensorFlow.js Layers format (model.json + weights)
import tensorflowjs as tfjs

TFJS_DIR = MODEL_DIR / f"tfjs_lower_body_{timestamp}"
TFJS_DIR.mkdir(parents=True, exist_ok=True)

print("🔄 Converting to TensorFlow.js (Layers format)...")
print("   Input (Keras .h5):", H5_PATH)
print("   Output folder:     ", TFJS_DIR)

tfjs.converters.save_keras_model(model, str(TFJS_DIR))

print("✅ TFJS conversion completed")
print("📄 Files created:")
for p in sorted(TFJS_DIR.iterdir()):
    print("  -", p.name)


/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


🔄 Converting to TensorFlow.js (Layers format)...
   Input (Keras .h5): /workspace/models/lower_body_mobilenetv2_20260202_130936.h5
   Output folder:      /workspace/models/tfjs_lower_body_20260202_130936
✅ TFJS conversion completed
📄 Files created:
  - group1-shard1of3.bin
  - group1-shard2of3.bin
  - group1-shard3of3.bin
  - model.json


In [27]:
# Cell 17: Quick verification of TFJS output files
model_json = TFJS_DIR / "model.json"
print("🔎 Verifying TFJS export...")
print("   model.json exists:", model_json.exists())

if not model_json.exists():
    raise FileNotFoundError(f"❌ model.json not found in {TFJS_DIR}")

# show small peek (first ~300 chars)
txt = model_json.read_text(encoding="utf-8")
print("✅ model.json looks readable")
print("   Preview:", txt[:300].replace("\n", " ") + " ...")


🔎 Verifying TFJS export...
   model.json exists: True
✅ model.json looks readable
   Preview: {"format": "layers-model", "generatedBy": "keras v2.15.0", "convertedBy": "TensorFlow.js Converter v4.22.0", "modelTopology": {"keras_version": "2.15.0", "backend": "tensorflow", "model_config": {"class_name": "Functional", "config": {"name": "model_1", "trainable": true, "layers": [{"class_name": " ...


In [28]:
# Cell 18: Final paths summary (copy these into your JS project)
print("\n" + "="*60)
print("✅ DONE: Artifacts ready for JavaScript")
print("="*60)

print("🧠 Keras model (.h5):", H5_PATH)
print("🧠 SavedModel dir:   ", SAVEDMODEL_DIR)
print("🌐 TFJS folder:      ", TFJS_DIR)
print("🏷️ Classes JSON:     ", CLASSES_JSON)



✅ DONE: Artifacts ready for JavaScript
🧠 Keras model (.h5): /workspace/models/lower_body_mobilenetv2_20260202_130936.h5
🧠 SavedModel dir:    /workspace/models/lower_body_mobilenetv2_savedmodel_20260202_130936
🌐 TFJS folder:       /workspace/models/tfjs_lower_body_20260202_130936
🏷️ Classes JSON:      /workspace/models/lower_body_classes_20260202_130936.json
